# Evaluation summary

Headline tables for global priors, preflop range posteriors (online), and player-θ actions (online), using `utils.eval` drivers.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

def _find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for x in (p, *p.parents):
        if (x / "runners" / "common.py").is_file():
            return x
    raise FileNotFoundError("Run from inside the repo (runners/common.py not found).")

REPO = _find_repo_root()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import json

import numpy as np

from utils.eval.repo_paths import default_global_priors_eval_session_paths
from utils.eval.global_priors_supervised import load_or_build_global_priors_supervised
from utils.eval.action_heads import (
    empirical_marginal,
    global_priors_three_head_predictions,
    remap_no_bet_labels,
    nll,
    mean_brier,
    top1_accuracy,
)
from utils.eval.preflop_range_posterior_eval import (
    em_and_online_hand_refs,
    compute_all_posterior_results,
    true_hand_nll,
    ranks_from_results,
    default_observer_map,
)
from utils.strength.preflop import all_169_classes
from utils.eval.player_thetas_eval import (
    load_global_betas,
    load_player_thetas,
    build_player_prediction_dict,
    nll as player_nll,
    heldout_gradient_norm_components,
)

train_file, eval_component_file, theta_file = default_global_priors_eval_session_paths(REPO)
bundle = load_or_build_global_priors_supervised(
    REPO,
    train_file=train_file,
    eval_component_file=eval_component_file,
    theta_file=theta_file,
    force_rebuild=False,
)
priors = json.loads((REPO / "artifacts" / "global_priors.json").read_text())
BETA_PRE, BETA_FACING, BETA_NO_BET = load_global_betas(priors)
thetas_payload = json.loads((REPO / "artifacts" / "player_thetas.json").read_text())
PLAYERS, THETA_PRE, THETA_POST = load_player_thetas(thetas_payload)
OBSERVER_OF = default_observer_map(PLAYERS)

In [ ]:
# 1) Global priors
y_n_train_local = remap_no_bet_labels(bundle.yn_train)
y_n_test_local = remap_no_bet_labels(bundle.yn_test)
marg_pre = empirical_marginal(bundle.y_pre_train, 3)
marg_facing = empirical_marginal(bundle.yf_train, 3)
marg_no_bet = empirical_marginal(y_n_train_local, 2)
preds = global_priors_three_head_predictions(
    beta_pre=BETA_PRE,
    beta_facing=BETA_FACING,
    beta_no_bet=BETA_NO_BET,
    X_pre_test=bundle.X_pre_test,
    Xf_test=bundle.Xf_test,
    Xn_test=bundle.Xn_test,
    marg_pre=marg_pre,
    marg_facing=marg_facing,
    marg_no_bet=marg_no_bet,
)
(Ppt, Pph, Ppm), (Pft, Pfh, Pfm), (Pnt, Pnh, Pnm) = preds
print(f"{'head':<8} {'model':<10} {'N':>4} {'NLL':>7} {'Brier':>7} {'top-1':>7}")
for head, y, seq in (
    ("preflop", bundle.y_pre_test, (("trained", Ppt), ("heuristic", Pph), ("marginal", Ppm))),
    ("facing", bundle.yf_test, (("trained", Pft), ("heuristic", Pfh), ("marginal", Pfm))),
    ("no_bet", y_n_test_local, (("trained", Pnt), ("heuristic", Pnh), ("marginal", Pnm))),
):
    for name, P in seq:
        print(f"{head:<8} {name:<10} {y.size:>4} {nll(P, y):>7.4f} {mean_brier(P, y):>7.4f} {top1_accuracy(P, y):>7.4f}")

In [ ]:
# 2) Range online
em_refs, online_refs = em_and_online_hand_refs(REPO)
RES = compute_all_posterior_results(
    PLAYERS, OBSERVER_OF, beta_pre=BETA_PRE, theta_pre=THETA_PRE,
    em_refs=em_refs, online_refs=online_refs,
)
ALL169 = all_169_classes()
IX = {h: i for i, h in enumerate(ALL169)}
print(f"{'player':<10} {'predictor':<11} {'NLL':>7} {'mean_rk':>8} {'top-10':>7} {'top-50':>7}")
for player in PLAYERS:
    for name in ("prior_only", "population", "player"):
        lst = RES[(player, "online", name)]
        r = ranks_from_results(lst, all_169=ALL169, index_of=IX)
        nv = true_hand_nll(lst)
        print(f"{player:<10} {name:<11} {nv:>7.3f} {r.mean():>8.2f} {(r<=10).mean():>7.3f} {(r<=50).mean():>7.3f}")

In [ ]:
# 3) Player θ online
PRED = build_player_prediction_dict(
    PLAYERS,
    B_pre=BETA_PRE,
    B_facing=BETA_FACING,
    B_no_bet=BETA_NO_BET,
    theta_pre=THETA_PRE,
    theta_post=THETA_POST,
    em_refs=em_refs,
    online_refs=online_refs,
)
print(f"{'player':<10} {'head':<8} {'N':>4} {'NLL_0':>7} {'NLL_θ':>7} {'ΔNLL':>+7} {'||g||':>8}")
for player in PLAYERS:
    for head in ("preflop", "facing", "no_bet"):
        P0, Pt, y = PRED[(player, "online", head)]
        if head == "preflop":
            th, k = THETA_PRE[player], 3
        elif head == "facing":
            th, k = THETA_POST[player], 3
        else:
            th, k = THETA_POST[player], 2
        g = heldout_gradient_norm_components(Pt, y, th, k)
        n0, nt = player_nll(P0, y), player_nll(Pt, y)
        print(f"{player:<10} {head:<8} {y.size:>4} {n0:>7.4f} {nt:>7.4f} {n0-nt:>+7.4f} {float(np.linalg.norm(g)):>8.5f}")

## Notes

Interpret the three blocks together: global prior quality, whether E-step ranges beat the dead-cards prior, and whether $\hat\theta$ moves online action likelihood / gradients.